In [25]:
#Imports
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

from sklearn.pipeline import Pipeline
from gensim.models import Word2Vec
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [12]:
# Load the dataset splits (train / validation / test)
dataset = load_dataset("AI-team-UoA/greek_legal_code")
train = dataset['train']
val   = dataset['validation']
test  = dataset['test']

# Extract texts and labels for each split
X_train, y_train = train['text'], train['label']
X_val,   y_val   = val['text'],   val['label']
X_test,  y_test  = test['text'],  test['label']

In [13]:
# Quick sanity-check of sizes
print("Train:", len(X_train), 
      "Val:",   len(X_val), 
      "Test:",  len(X_test))

Train: 28536 Val: 9511 Test: 9516


In [14]:
# === Combined BoW → TF–IDF → SVM pipeline ===
pipeline_svm = Pipeline([
    # 1) Bag-of-Words: extract uni- and bi-gram counts (max 50k features)
    ('count', CountVectorizer(ngram_range=(1,2), max_features=50000, lowercase=True)),
    # 2) Re-weight to TF–IDF
    ('tfidf', TfidfTransformer()),
    # 3) Linear Support Vector Classifier
    ('clf', LinearSVC(C=1.0, max_iter=10000, random_state=42))
])

In [15]:
print("=== SVM με BoW + TF-IDF ===")
pipeline_svm.fit(X_train, y_train)
y_val_pred_svm  = pipeline_svm.predict(X_val)
y_test_pred_svm = pipeline_svm.predict(X_test)

print("Validation Report:")
print(classification_report(y_val, y_val_pred_svm, zero_division=0))
print("Test Report:")
print(classification_report(y_test, y_test_pred_svm, zero_division=0))

=== SVM με BoW + TF-IDF ===
Validation Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82       187
           1       0.83      0.92      0.87       389
           2       0.91      0.78      0.84        79
           3       0.84      0.84      0.84       244
           4       0.89      0.89      0.89       346
           5       0.84      0.81      0.82       108
           6       0.92      0.88      0.90       224
           7       0.88      0.72      0.79       111
           8       0.84      0.70      0.77        91
           9       0.82      0.75      0.78        71
          10       0.94      0.83      0.88       160
          11       0.83      0.86      0.85       184
          12       0.92      0.82      0.87        83
          13       0.87      0.87      0.87       152
          14       0.93      0.92      0.92       169
          15       0.85      0.79      0.82       155
          16       0.82      0.79 

In [16]:
# 1) Pure BoW + SVM
pipeline_bow = Pipeline([
    # Bag-of-Words counts only
    ('count', CountVectorizer(ngram_range=(1,2), max_features=50000, lowercase=True)),
    # Linear SVM on raw counts
    ('clf',   LinearSVC(C=1.0, max_iter=10000, random_state=42))
])

In [17]:
print("=== SVM με Μόνο BoW ===")
pipeline_bow.fit(X_train, y_train)
y_val_pred_bow  = pipeline_bow.predict(X_val)
y_test_pred_bow = pipeline_bow.predict(X_test)

print("Validation Report:")
print(classification_report(y_val,  y_val_pred_bow,  zero_division=0))
print("Test Report:")
print(classification_report(y_test, y_test_pred_bow, zero_division=0))

=== SVM με Μόνο BoW ===
Validation Report:
              precision    recall  f1-score   support

           0       0.73      0.75      0.74       187
           1       0.78      0.84      0.81       389
           2       0.73      0.73      0.73        79
           3       0.72      0.75      0.73       244
           4       0.78      0.80      0.79       346
           5       0.74      0.72      0.73       108
           6       0.85      0.83      0.84       224
           7       0.71      0.59      0.64       111
           8       0.72      0.64      0.68        91
           9       0.72      0.66      0.69        71
          10       0.91      0.84      0.87       160
          11       0.78      0.77      0.77       184
          12       0.87      0.73      0.80        83
          13       0.82      0.80      0.81       152
          14       0.86      0.85      0.85       169
          15       0.72      0.77      0.74       155
          16       0.76      0.68     

In [18]:
# 2) Pure TF–IDF + SVM
pipeline_tfidf = Pipeline([
    # TF–IDF vectorizer (combines counting + reweighting in one step)
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000, lowercase=True)),
    # Linear SVM on TF–IDF features
    ('clf',   LinearSVC(C=1.0, max_iter=10000, random_state=42))
])

In [19]:
print("=== SVM με Μόνο TF–IDF ===")
pipeline_tfidf.fit(X_train, y_train)
y_val_pred_tfidf  = pipeline_tfidf.predict(X_val)
y_test_pred_tfidf = pipeline_tfidf.predict(X_test)

print("Validation Report:")
print(classification_report(y_val,  y_val_pred_tfidf,  zero_division=0))
print("Test Report:")
print(classification_report(y_test, y_test_pred_tfidf, zero_division=0))

=== SVM με Μόνο TF–IDF ===
Validation Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82       187
           1       0.83      0.92      0.87       389
           2       0.91      0.78      0.84        79
           3       0.84      0.84      0.84       244
           4       0.89      0.89      0.89       346
           5       0.84      0.81      0.82       108
           6       0.92      0.88      0.90       224
           7       0.88      0.72      0.79       111
           8       0.84      0.70      0.77        91
           9       0.82      0.75      0.78        71
          10       0.94      0.83      0.88       160
          11       0.83      0.86      0.85       184
          12       0.92      0.82      0.87        83
          13       0.87      0.87      0.87       152
          14       0.93      0.92      0.92       169
          15       0.85      0.79      0.82       155
          16       0.82      0.79  

In [20]:
# 1) Tokenization
docs_train = [text.split() for text in X_train]
docs_val   = [text.split() for text in X_val]
docs_test  = [text.split() for text in X_test]

In [21]:
w2v = Word2Vec(
    sentences=docs_train,   # tokenized training documents
    vector_size=400,        # dimensionality of word vectors
    window=15,              # context window size
    min_count=1,            # include all words
    sg=1,                   # use skip-gram architecture
    negative=5,             # number of negative samples
    sample=1e-5,            # downsample frequent words
    epochs=20,              # number of training epochs
    workers=4,              # parallel worker threads
    seed=42                 # random seed for reproducibility
)
# build the vocabulary and then train
w2v.build_vocab(docs_train)
w2v.train(
    docs_train,
    total_examples=w2v.corpus_count,
    epochs=w2v.epochs
)


(146985220, 342684040)

In [22]:
def embed_doc(tokens):
    """
    Given a list of tokens, compute a 800-dim embedding by
    concatenating the mean and max over their Word2Vec vectors.
    """
    vecs = np.array([w2v.wv[w] for w in tokens if w in w2v.wv])
    if vecs.size == 0:
        return np.zeros(800) # empty document
    return np.hstack([vecs.mean(axis=0), vecs.max(axis=0)])

In [23]:
# embed all splits
X_train_vec = np.vstack([embed_doc(d) for d in docs_train])
X_val_vec   = np.vstack([embed_doc(d) for d in docs_val])
X_test_vec  = np.vstack([embed_doc(d) for d in docs_test])

In [26]:
# 4) Standardize embeddings for both classifiers
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train_vec)
X_val_s   = scaler.transform(  X_val_vec)
X_test_s  = scaler.transform(  X_test_vec)

In [27]:
# 5a) Logistic Regression on W2V embeddings
lr = LogisticRegression(
    C=0.01,
    penalty='l2',
    solver='lbfgs',           # fast on dense data
    max_iter=1000,
    tol=1e-3,                 # reasonable convergence
    class_weight='balanced',
    multi_class='multinomial',
    random_state=42,
    n_jobs=-1
)
lr.fit(X_train_s, y_train)

c:\Users\perik\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(C=0.01, class_weight='balanced', max_iter=1000,
                   multi_class='multinomial', n_jobs=-1, random_state=42,
                   tol=0.001)

In [28]:
for split, X, y in [
    ("Validation", X_val_s, y_val),
    ("Test",       X_test_s, y_test),
]:
    preds = lr.predict(X)
    acc  = accuracy_score(y, preds)
    prec = precision_score(y, preds, average='weighted', zero_division=0)
    rec  = recall_score(y, preds, average='weighted', zero_division=0)
    f1   = f1_score(y, preds, average='weighted', zero_division=0)

    print(f"\n{split} — Logistic Regression + W2V")
    print(f"  Accuracy:            {acc:.3f}")
    print(f"  Precision (weighted):{prec:.3f}")
    print(f"  Recall (weighted):   {rec:.3f}")
    print(f"  F1-score (weighted): {f1:.3f}")
    print(classification_report(y, preds, zero_division=0))


Validation — Logistic Regression + W2V
  Accuracy:            0.766
  Precision (weighted):0.777
  Recall (weighted):   0.766
  F1-score (weighted): 0.770
              precision    recall  f1-score   support

           0       0.76      0.68      0.72       187
           1       0.82      0.73      0.77       389
           2       0.62      0.73      0.67        79
           3       0.70      0.69      0.70       244
           4       0.81      0.78      0.79       346
           5       0.68      0.75      0.71       108
           6       0.93      0.87      0.90       224
           7       0.64      0.81      0.71       111
           8       0.65      0.70      0.67        91
           9       0.67      0.75      0.71        71
          10       0.61      0.69      0.65       160
          11       0.78      0.78      0.78       184
          12       0.84      0.84      0.84        83
          13       0.72      0.77      0.74       152
          14       0.72      0.82

In [31]:
mlp = MLPClassifier(
    hidden_layer_sizes=(1024, 512, 256),   # deeper network
    activation='relu',                     # keep ReLU for speed + sparsity
    solver='adam',                         
    alpha=1e-5,                            # lighter regularization
    batch_size=256,                        # larger batches for stable updates
    learning_rate='adaptive',              # cut lr when plateauing
    learning_rate_init=5e-4,               # slightly higher init lr
    max_iter=500,                          # allow more epochs to converge
    tol=1e-4,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,                   
    random_state=42,
    verbose=True                          
)

mlp.fit(X_train_s, y_train)

Iteration 1, loss = 1.88981075
Validation score: 0.692011
Iteration 2, loss = 0.78307201
Validation score: 0.751226
Iteration 3, loss = 0.50052516
Validation score: 0.762789
Iteration 4, loss = 0.32285969
Validation score: 0.771899
Iteration 5, loss = 0.21256009
Validation score: 0.769096
Iteration 6, loss = 0.13955145
Validation score: 0.782411
Iteration 7, loss = 0.08610206
Validation score: 0.779608
Iteration 8, loss = 0.05999716
Validation score: 0.786265
Iteration 9, loss = 0.03995168
Validation score: 0.786966
Iteration 10, loss = 0.04094009
Validation score: 0.784513
Iteration 11, loss = 0.01669721
Validation score: 0.790119
Iteration 12, loss = 0.02363350
Validation score: 0.781009
Iteration 13, loss = 0.03779113
Validation score: 0.789769
Iteration 14, loss = 0.03503734
Validation score: 0.772249
Iteration 15, loss = 0.06371532
Validation score: 0.757884
Iteration 16, loss = 0.07155073
Validation score: 0.768045
Iteration 17, loss = 0.03508159
Validation score: 0.776104
Iterat

MLPClassifier(alpha=1e-05, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(1024, 512, 256), learning_rate='adaptive',
              learning_rate_init=0.0005, max_iter=500, n_iter_no_change=20,
              random_state=42, verbose=True)

In [32]:
for split, X, y in [
    ("Validation", X_val_s, y_val),
    ("Test",       X_test_s, y_test),
]:
    preds = mlp.predict(X)
    acc  = accuracy_score(y, preds)
    prec = precision_score(y, preds, average='weighted', zero_division=0)
    rec  = recall_score(y, preds, average='weighted', zero_division=0)
    f1   = f1_score(y, preds, average='weighted', zero_division=0)

    print(f"\n{split} — MLP + W2V")
    print(f"  Accuracy:            {acc:.3f}")
    print(f"  Precision (weighted):{prec:.3f}")
    print(f"  Recall (weighted):   {rec:.3f}")
    print(f"  F1-score (weighted): {f1:.3f}")
    print(classification_report(y, preds, zero_division=0))


Validation — MLP + W2V
  Accuracy:            0.774
  Precision (weighted):0.780
  Recall (weighted):   0.774
  F1-score (weighted): 0.774
              precision    recall  f1-score   support

           0       0.80      0.68      0.74       187
           1       0.81      0.78      0.79       389
           2       0.69      0.65      0.67        79
           3       0.66      0.72      0.69       244
           4       0.76      0.83      0.79       346
           5       0.75      0.75      0.75       108
           6       0.93      0.85      0.89       224
           7       0.77      0.64      0.70       111
           8       0.76      0.65      0.70        91
           9       0.61      0.65      0.63        71
          10       0.75      0.54      0.63       160
          11       0.78      0.77      0.77       184
          12       0.85      0.75      0.79        83
          13       0.67      0.84      0.74       152
          14       0.80      0.80      0.80      